# JMangaTranslator-Fast v1

Japanese → Simplified Chinese translation of manga speech bubbles, one bubble at a time.
日译简中漫画对话框翻译，一次翻译一个框。

- **Colab**: Runtime → Change runtime type → **T4 GPU**.
- **Kaggle**: Settings → Accelerator → **GPU T4**, and turn **Internet** on.
- Without a GPU the notebook runs on the CPU with ONNX Runtime (slower). 没有 GPU 时改用 CPU 上的 ONNX Runtime（较慢）。

Model card: https://huggingface.co/muscgab/JMangaTranslator-Fast

In [ ]:
# 1. Install dependencies (PyTorch is preinstalled on Colab and Kaggle)
import subprocess, sys
import torch

GPU = torch.cuda.is_available()
pkgs = ["transformers>=5.14", "safetensors", "sentencepiece", "tokenizers", "huggingface_hub"]
if not GPU:
    pkgs.append("onnxruntime")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
print("GPU:", torch.cuda.get_device_name(0) if GPU else "none, using the CPU (ONNX Runtime)")

In [ ]:
# 2. Download only the files this runtime needs (about 1.5 GB)
from huggingface_hub import snapshot_download

files = ["config.json", "vocab.json", "joint.model", "tokenizer/*", "jmt_fast/*", "translate.py"]
files += ["model.safetensors", "norm_scales.json"] if GPU else ["onnx/*"]
path = snapshot_download("muscgab/JMangaTranslator-Fast", allow_patterns=files)
print(path)

In [ ]:
# 3. Load the model once (CUDA Graphs fp16 on a GPU, ONNX Runtime on the CPU)
import time
sys.path.insert(0, path)
from jmt_fast import load, pick

t = time.perf_counter()
tr = load(path)
tr.translate("こんにちは")   # warm-up: the first call is slower
print(f"backend: {pick()}, loaded in {time.perf_counter() - t:.1f} s")

In [ ]:
# 4. Translate. Each call is one speech bubble; remove line breaks inside a bubble first.
bubbles = [
    "堪忍袋の緒が切れた！",
    "お前、まだそんなこと言ってるのか？",
    "今日こそ決着をつけてやる！",
    "べ、別にあんたのためじゃないんだからね！",
    "腹減ったなあ……",
]
for b in bubbles:
    t = time.perf_counter()
    out = tr.translate(b)
    print(f"{1000 * (time.perf_counter() - t):6.1f} ms  {b}  →  {out}")

In [ ]:
# 5. Your own text
print(tr.translate("ここに日本語のセリフを入力してください"))

In [ ]:
# 6. A file with one bubble per line: input.txt → output.txt
from pathlib import Path

src = Path("input.txt")
if src.exists():
    lines = src.read_text(encoding="utf-8").splitlines()
    Path("output.txt").write_text("\n".join(tr.translate(x) if x.strip() else "" for x in lines) + "\n", encoding="utf-8")
    print(f"translated {len(lines)} lines → output.txt")
else:
    print("Upload input.txt (one bubble per line) and run this cell again.")